In [1]:
import os

import mlflow
import polars as pl
import plotly.graph_objects as go
import plotly.colors as pc

from paretodo.dashboard.utils.types_and_constants import LOCAL_TEMP_DIR

from setup.plotly_theme import register_theme

register_theme()

PDF_FOLDER = ".local/pdf"
os.makedirs(PDF_FOLDER, exist_ok=True)

# Colour-blind-safe palette: Okabe-Ito plus three Paul Tol colours, LHS in black
color_dict = {
    "pareto_expert": "#0072B2",
    "pareto_expert_80": "#56B4E9",
    "pareto_expert_95": "#CCBB44",
    "pareto_noob": "#D55E00",
    "pareto_utopia": "#009E73",
    "pareto_explore": "#CC79A7",
    "pareto_exploit": "#E69F00",
    "bo_EI": "#EE3377",
    "bo_UCB": "#332288",
    "lhs": "#000000",
}

symbol_dict = {
    "pareto_expert": "circle",
    "pareto_expert_80": "square",
    "pareto_expert_95": "diamond",
    "pareto_noob": "cross",
    "pareto_utopia": "triangle-up",
    "pareto_explore": "triangle-down",
    "pareto_exploit": "star",
    "bo_EI": "pentagon",
    "bo_UCB": "hexagon",
    "lhs": "x",
}

MARKER_SIZE = 14  # markers on plotted lines / points
LEGEND_MARKER_SIZE = 20  # markers in the legend

/Users/ss15425/Documents/00_Git_Repositories/pfgs-optimization/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "Rev-Benchmark_Opt-15-09-2026"

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = client.search_runs(experiment_ids=[experiment.experiment_id])

pareto_expert_df = pl.DataFrame()
pareto_noob_df = pl.DataFrame()
pareto_utopia_df = pl.DataFrame()
pareto_explore_df = pl.DataFrame()
pareto_exploit_df = pl.DataFrame()
bo_EI_df = pl.DataFrame()
bo_UCB_df = pl.DataFrame()
lhs_df = pl.DataFrame()


for i, run in enumerate(runs):
    algo_name = run.data.tags.get("Algorithm", None)
    if algo_name == "lhs":
        name = "lhs"
    elif algo_name == "bo":
        if "bo_config" in run.data.params:
            bo_config = run.data.params.get("bo_config", None)
            acquisition = bo_config.split("acquisition_function': '")[1].split("',")[0]
            name = f"bo_{acquisition}"
    else:
        name = run.data.params.get("sampling_mode", None)
        if not name:
            raise ValueError(f"Sampling mode not found for run {run.info.run_id}")
    if name == "automatic":
        name = "pareto_utopia"
    metric = client.get_metric_history(run.info.run_id, "best_product_concentration")
    if i == 0:
        data_df = pl.DataFrame({"step": [m.step for m in metric]})
    if name == "pareto_expert":
        pareto_expert_df = pareto_expert_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
    elif name == "pareto_noob":
        pareto_noob_df = pareto_noob_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
    elif name == "pareto_utopia":
        pareto_utopia_df = pareto_utopia_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
    elif name == "pareto_explore":
        pareto_explore_df = pareto_explore_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
    elif name == "pareto_exploit":
        pareto_exploit_df = pareto_exploit_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
    elif name == "bo_EI":
        bo_EI_df = bo_EI_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
    elif name == "bo_UCB":
        bo_UCB_df = bo_UCB_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
    elif name == "lhs":
        lhs_df = lhs_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
    else:
        print(f"Unknown sampling mode: {name}")


In [3]:
algorithms = [
    "pareto_expert",
    "pareto_noob",
    "pareto_utopia",
    "pareto_explore",
    "pareto_exploit",
    "bo_EI",
    "bo_UCB",
    "lhs",
]
name_dict = {
    "pareto_expert": "Pareto Expert",
    "pareto_expert_80" : "Pareto Expert 80%",
    "pareto_expert_95" : "Pareto Expert 95%",
    "pareto_noob": "Pareto Adversarial",
    "pareto_utopia": "Pareto Utopia",
    "pareto_explore": "Pareto Explore",
    "pareto_exploit": "Pareto Exploit",
    "bo_EI": "BO EI",
    "bo_UCB": "BO UCB",
    "lhs": "LHS",
}
mean_data_df = pl.DataFrame()
mean_data_df = mean_data_df.with_columns(pl.Series("Experiments", range(len(pareto_expert_df)))+11)
for algo in algorithms:
    mean_data_df = mean_data_df.with_columns(
        pl.Series(algo, eval(f"{algo}_df").mean_horizontal())
    )


std_data_df = pl.DataFrame()
std_data_df = std_data_df.with_columns(pl.Series("Experiments", range(len(pareto_expert_df)))+11)
for algo in algorithms:
    std_data_df = std_data_df.with_columns(
        pl.Series(algo, eval(f"{algo}_df").select(pl.concat_list(pl.all()).list.std().alias(algo)))
    )

mean_data_df.write_csv(os.path.join(LOCAL_TEMP_DIR, "mean_data_persona.csv"))
std_data_df.write_csv(os.path.join(LOCAL_TEMP_DIR, "std_data_persona.csv"))

In [4]:

algos = [
    "pareto_expert",
    "pareto_noob",
    # "pareto_utopia",
    "pareto_explore",
    "pareto_exploit",
]

def _hex_to_rgba(hex_color, alpha):
    r, g, b = pc.hex_to_rgb(hex_color)
    return f"rgba({r},{g},{b},{alpha})"


fig = go.Figure()

for i, algo in enumerate(algos):
    color = color_dict[algo]

    upper = mean_data_df[algo] + std_data_df[algo]
    lower = mean_data_df[algo] - std_data_df[algo]

    fig.add_trace(
        go.Scatter(
            x=mean_data_df["Experiments"],
            y=upper,
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
            legendgroup=algo,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=mean_data_df["Experiments"],
            y=lower,
            line=dict(width=0),
            fill="tonexty",
            fillcolor=_hex_to_rgba(color, 0.2),
            showlegend=False,
            hoverinfo="skip",
            legendgroup=algo,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=mean_data_df["Experiments"],
            y=mean_data_df[algo],
            name=name_dict[algo],
            mode="lines+markers",
            line=dict(width=2, color=color),
            marker=dict(size=MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
            showlegend=False,
            legendgroup=algo,
        )
    )
    # legend-only trace so the legend line is thicker than the plotted line
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            name=name_dict[algo],
            mode="lines+markers",
            line=dict(width=6, color=color),
            marker=dict(size=LEGEND_MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
            legendgroup=algo,
        )
    )

fig.update_layout(legend=dict(bgcolor="rgba(255,255,255,0.5)",groupclick="togglegroup",font=dict(size=30), x=0.02, y=1,),margin=dict(l=130, r=20, t=100, b=120), width=1000, height=800, title="Mean and One Standard Deviation of Best Product Concentration", title_font_size=30)
fig.update_xaxes(title_text="Number of Experiments", title_font_size=30, tickfont=dict(size=30), range=[10,41])
fig.update_yaxes(title_text="Best Product Concentration", title_font_size=30, tickfont=dict(size=30),dtick=200)
fig.show()
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.write_image(f"{PDF_FOLDER}/best_objective_pfgs.pdf")


2026-10-05 15:52:25,130 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 15:52:25,132 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 15:52:25,133 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpkunjvfd7.
2026-10-05 15:52:25,134 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 15:52:25,135 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp8b88_qzb.
2026-10-05 15:52:25,135 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp8b88_qzb
2026-10-05 15:52:26,189 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpkunjvfd7/index.html
2026-10-05 15:52:26,799 - kaleido.kaleido - INFO - Getting tab from queue 

In [5]:
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "Rev-Benchmark_Opt-15-09-2026"
X_RANGE = [0, 3000]
Y_RANGE = [0, 1000]

pfgs_runs = {
    "pareto_expert": "Pareto_Expert",
    "pareto_noob": "Pareto_Noob",
    # "pareto_utopia": "Pareto_Utopia",
    "pareto_explore": "Pareto_Explore",
    "pareto_exploit": "Pareto_Exploit",
}

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = client.search_runs(experiment_ids=[experiment.experiment_id])


def load_pareto_front(run_name, iteration):
    selected_run = next(r for r in runs if r.info.run_name == run_name)
    temp_dir = os.path.join(LOCAL_TEMP_DIR, run_name)
    os.makedirs(temp_dir, exist_ok=True)
    client.download_artifacts(run_id=selected_run.info.run_id, path=iteration, dst_path=temp_dir)
    return pl.read_csv(f"{temp_dir}/{iteration}/pareto_front_predictions.csv")

def plot_pareto_fronts(iteration):
    it = int(iteration.split("_")[1])
    fig = go.Figure()
    for algo, run_name in pfgs_runs.items():
        pareto_df = load_pareto_front(run_name, iteration)
        # pareto_df = pareto_df.write_csv(os.path.join(LOCAL_TEMP_DIR, f"pareto_front_{algo}_{iteration}.csv"))
        if pareto_df is None:
            print(f"No Pareto front data found for {algo} at iteration {iteration}")
            continue
        fig.add_trace(
            go.Scatter(
                x=pareto_df["obj:max(μ(product_concentration))"],
                y=pareto_df["obj:max(σ(product_concentration))"],
                mode="markers",
                name=f"{name_dict[algo]}",
                marker=dict(
                    size=MARKER_SIZE,
                    color=color_dict[algo],
                    line=dict(width=2, color="black"),
                    symbol=symbol_dict[algo],
                ),
            )
        )
    fig.update_layout(
        margin=dict(l=130, r=20, t=70, b=120),
        font=dict(size=30),
        title=f"Comparison of Pareto Front Predictions at Iteration {it}",
        title_font_size=30,
        width=1000,
        height=800,
        legend=dict(bgcolor="rgba(255,255,255,0.5)", font=dict(size=30), x=0.02, y=0.03, xanchor="left", yanchor="bottom"),
    )
    fig.update_xaxes(title_text="Max Mean Product Concentration", title_font_size=30, tickfont=dict(size=30), range=X_RANGE)
    fig.update_yaxes(title_text="Max Std Dev Product Concentration", title_font_size=30, tickfont=dict(size=30), range=Y_RANGE)
    return fig


2026-10-05 15:52:27,164 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 15:52:27,165 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


In [6]:
fig = plot_pareto_fronts("Iteration_010")
fig.write_image(f"{PDF_FOLDER}/pareto_fronts_pfgs_010.pdf")
fig.show()

2026-10-05 15:52:27,329 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 15:52:27,330 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 15:52:27,331 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp1mblhojv.
2026-10-05 15:52:27,331 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 15:52:27,332 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp73pz4ean.
2026-10-05 15:52:27,332 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp73pz4ean
2026-10-05 15:52:27,766 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp1mblhojv/index.html
2026-10-05 15:52:28,357 - kaleido.kaleido - INFO - Getting tab from queue 

In [7]:
fig = plot_pareto_fronts("Iteration_020")
fig.write_image(f"{PDF_FOLDER}/pareto_fronts_pfgs_020.pdf")

fig.show()

2026-10-05 15:52:28,685 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 15:52:28,686 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.
2026-10-05 15:52:28,763 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 15:52:28,764 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 15:52:28,765 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp93df8av4.
2026-10-05 15:52:28,765 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 15:52:28,766 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp4cev8dmn.
2026-10-05 15:52:28,766 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp4cev8dmn
2026-10-05 15:52:29,198 - kaleido

In [8]:
fig = plot_pareto_fronts("Iteration_030")
fig.write_image(f"{PDF_FOLDER}/pareto_fronts_pfgs_030.pdf")
fig.show()

2026-10-05 15:52:30,274 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 15:52:30,275 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.
2026-10-05 15:52:30,279 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 15:52:30,280 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 15:52:30,281 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpo4dnm0ka.
2026-10-05 15:52:30,282 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 15:52:30,282 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmppiwcki55.
2026-10-05 15:52:30,283 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmppiwcki55
2026-10-05 15:52:30,697 - kaleido

In [9]:
algos = [
    "pareto_expert",
    "pareto_exploit",
    "bo_EI",
    "bo_UCB",
    "lhs",
]

def _hex_to_rgba(hex_color, alpha):
    r, g, b = pc.hex_to_rgb(hex_color)
    return f"rgba({r},{g},{b},{alpha})"


fig = go.Figure()

for i, algo in enumerate(algos):
    color = color_dict[algo]

    upper = mean_data_df[algo] + std_data_df[algo]
    lower = mean_data_df[algo] - std_data_df[algo]

    if algo == "lhs":
        fig.add_trace(
            go.Scatter(
                x=mean_data_df["Experiments"],
                y=mean_data_df[algo],
                name=name_dict[algo],
                mode="markers",
                marker=dict(size=MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
                error_y=dict(
                    type="data",
                    array=std_data_df[algo],
                    visible=True,
                    color=color,
                    thickness=2,
                    width=10,
                ),
                legendgroup=algo,
            )
        )
    else:
        fig.add_trace(
            go.Scatter(
                x=mean_data_df["Experiments"],
                y=upper,
                line=dict(width=0),
                showlegend=False,
                hoverinfo="skip",
                legendgroup=algo,
            )
        )
        fig.add_trace(
            go.Scatter(
                x=mean_data_df["Experiments"],
                y=lower,
                line=dict(width=0),
                fill="tonexty",
                fillcolor=_hex_to_rgba(color, 0.2),
                showlegend=False,
                hoverinfo="skip",
                legendgroup=algo,
            )
        )
        fig.add_trace(
            go.Scatter(
                x=mean_data_df["Experiments"],
                y=mean_data_df[algo],
                name=name_dict[algo],
                mode="lines+markers",
                line=dict(width=2, color=color),
                marker=dict(size=MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
                showlegend=False,
                legendgroup=algo,
            )
        )
        # legend-only trace so the legend line is thicker than the plotted line
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                name=name_dict[algo],
                mode="lines+markers",
                line=dict(width=6, color=color),
                marker=dict(size=LEGEND_MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
                legendgroup=algo,
            )
        )

fig.update_layout(legend=dict(bgcolor="rgba(255,255,255,0.5)",groupclick="togglegroup",font=dict(size=30), x=0.02, y=1,),margin=dict(l=130, r=20, t=100, b=120), width=1000, height=800, title="Mean and One Standard Deviation of Best Product Concentration", title_font_size=30)
fig.update_xaxes(title_text="Number of Experiments", title_font_size=30, tickfont=dict(size=30), range=[10,41])
fig.update_yaxes(title_text="Best Product Concentration", title_font_size=30, tickfont=dict(size=30),dtick=200)
fig.show()
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.write_image(f"{PDF_FOLDER}/best_objective_bo.pdf")


2026-10-05 15:52:31,542 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 15:52:31,543 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 15:52:31,558 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 15:52:31,559 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 15:52:31,559 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpk9q8wyoi.
2026-10-05 15:52:31,560 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 15:52:31,561 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpmqdupggo.
2026-10-05 15:52:31,561 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpmqdupggo
2026-10-05 15:52:31,990 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpk9q8wyoi/index.html
2026-10-05 15:52:32,567 - kaleido.kaleido - INFO - Getting tab from queue 

## Constrain Case 

In [10]:
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "Rev-Benchmark_OptC-15-09-2026"

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = client.search_runs(experiment_ids=[experiment.experiment_id])

pareto_expert_80_df = pl.DataFrame()
pareto_expert_80_c_df = pl.DataFrame()
pareto_expert_95_df = pl.DataFrame()
pareto_expert_95_c_df = pl.DataFrame()
pareto_utopia_df = pl.DataFrame()
pareto_utopia_c_df = pl.DataFrame()
pareto_exploit_df = pl.DataFrame()
pareto_exploit_c_df = pl.DataFrame()
bo_EI_df = pl.DataFrame()
bo_EI_c_df = pl.DataFrame()
bo_UCB_df = pl.DataFrame()
bo_UCB_c_df = pl.DataFrame()
lhs_df = pl.DataFrame()
lhs_c_df = pl.DataFrame()


for i, run in enumerate(runs):
    algo_name = run.data.tags.get("Algorithm", None)
    if algo_name == "lhs":
        name = "lhs"
    elif algo_name == "bo":
        bo_config = run.data.params.get("bo_config", None)
        acquisition = bo_config.split("acquisition_function': '")[1].split("',")[0]
        name = f"bo_{acquisition}"
    else:
        name = run.data.params.get("sampling_mode", None)

    if name == "automatic":
        name = "pareto_utopia"
    metric = client.get_metric_history(run.info.run_id, "best_product_concentration")
    c_metric = client.get_metric_history(run.info.run_id, "constraint_violations_purity")
    if i == 0:
        data_df = pl.DataFrame({"step": [m.step for m in metric]})
    if name == "pareto_expert":
        discard_values = run.data.params.get("discard_values", None)
        if discard_values and "0.8" in discard_values:
            name = "pareto_expert_80"
            pareto_expert_80_df = pareto_expert_80_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
            pareto_expert_80_c_df = pareto_expert_80_c_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in c_metric]))
        elif discard_values and "0.95" in discard_values:
            name = "pareto_expert_95"
            pareto_expert_95_df = pareto_expert_95_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
            pareto_expert_95_c_df = pareto_expert_95_c_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in c_metric]))
    elif name == "pareto_utopia":
        pareto_utopia_df = pareto_utopia_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
        pareto_utopia_c_df = pareto_utopia_c_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in c_metric]))
    elif name == "pareto_exploit":
        pareto_exploit_df = pareto_exploit_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
        pareto_exploit_c_df = pareto_exploit_c_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in c_metric]))
    elif name == "bo_EI":
        bo_EI_df = bo_EI_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
        bo_EI_c_df = bo_EI_c_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in c_metric]))
    elif name == "bo_UCB":
        bo_UCB_df = bo_UCB_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
        bo_UCB_c_df = bo_UCB_c_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in c_metric]))
    elif name == "lhs":
        lhs_df = lhs_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in metric]))
        lhs_c_df = lhs_c_df.with_columns(pl.Series(f"{name}_{i}", [m.value for m in c_metric]))
    else:
        print(f"Unknown sampling mode: {name}")

2026-10-05 15:52:32,833 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 15:52:32,833 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


In [11]:
algorithms = [
    "pareto_expert_80",
    "pareto_expert_95",
    # "pareto_utopia",
    # "pareto_exploit",
    "bo_EI",
    "bo_UCB",
    "lhs"
]

mean_data_df = pl.DataFrame()
mean_violations_df = pl.DataFrame()
mean_data_df = mean_data_df.with_columns(pl.Series("Experiments", range(len(pareto_expert_df)))+11)
mean_violations_df = mean_violations_df.with_columns(pl.Series("Experiments", range(len(pareto_expert_80_df)))+11)
for algo in algorithms:
    print(f"Calculating mean for {algo}")
    mean_data_df = mean_data_df.with_columns(
        pl.Series(algo, eval(f"{algo}_df").mean_horizontal())
    )
    mean_violations_df = mean_violations_df.with_columns(
        pl.Series(algo, eval(f"{algo}_c_df").mean_horizontal())
    )


std_data_df = pl.DataFrame()
std_violations_df = pl.DataFrame()
std_data_df = std_data_df.with_columns(pl.Series("Experiments", range(len(pareto_expert_df)))+11)
std_violations_df = std_violations_df.with_columns(pl.Series("Experiments", range(len(pareto_expert_80_df)))+11)
for algo in algorithms:
    std_data_df = std_data_df.with_columns(
        pl.Series(algo, eval(f"{algo}_df").select(pl.concat_list(pl.all()).list.std().alias(algo)))
    )
    std_violations_df = std_violations_df.with_columns(
        pl.Series(algo, eval(f"{algo}_c_df").select(pl.concat_list(pl.all()).list.std().alias(algo)))
    )

mean_data_df.write_csv(os.path.join(LOCAL_TEMP_DIR, "mean_data_persona_c.csv"))
std_data_df.write_csv(os.path.join(LOCAL_TEMP_DIR, "std_data_persona_c.csv"))
mean_violations_df.write_csv(os.path.join(LOCAL_TEMP_DIR, "mean_violations_persona_c.csv"))
std_violations_df.write_csv(os.path.join(LOCAL_TEMP_DIR, "std_violations_persona_c.csv"))

Calculating mean for pareto_expert_80
Calculating mean for pareto_expert_95
Calculating mean for bo_EI
Calculating mean for bo_UCB
Calculating mean for lhs


In [12]:

def _hex_to_rgba(hex_color, alpha):
    r, g, b = pc.hex_to_rgb(hex_color)
    return f"rgba({r},{g},{b},{alpha})"


fig = go.Figure()

for i, algo in enumerate(algorithms):
    color = color_dict[algo]

    upper = mean_data_df[algo] + std_data_df[algo]
    lower = mean_data_df[algo] - std_data_df[algo]
    if algo == "lhs":
        fig.add_trace(
            go.Scatter(
                x=mean_data_df["Experiments"],
                y=mean_data_df[algo],
                name=name_dict[algo],
                mode="markers",
                marker=dict(size=MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
                error_y=dict(
                    type="data",
                    array=std_data_df[algo],
                    visible=True,
                    color=color,
                    thickness=2,
                    width=10,
                ),
                legendgroup=algo,
            )
        )
    else:
        fig.add_trace(
            go.Scatter(
                x=mean_data_df["Experiments"],
                y=upper,
                line=dict(width=0),
                showlegend=False,
                hoverinfo="skip",
                legendgroup=algo,
            )
        )
        fig.add_trace(
            go.Scatter(
                x=mean_data_df["Experiments"],
                y=lower,
                line=dict(width=0),
                fill="tonexty",
                fillcolor=_hex_to_rgba(color, 0.2),
                showlegend=False,
                hoverinfo="skip",
                legendgroup=algo,
            )
        )
        fig.add_trace(
            go.Scatter(
                x=mean_data_df["Experiments"],
                y=mean_data_df[algo],
                name=name_dict[algo],
                mode="lines+markers",
                line=dict(width=2, color=color),
                marker=dict(size=MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
                showlegend=False,
                legendgroup=algo,
            )
        )
        # legend-only trace so the legend line is thicker than the plotted line
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                name=name_dict[algo],
                mode="lines+markers",
                line=dict(width=6, color=color),
                marker=dict(size=LEGEND_MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
                legendgroup=algo,
            )
        )

fig.update_layout(legend=dict(bgcolor="rgba(255,255,255,0.5)", groupclick="togglegroup",font=dict(size=30), x=0.02, y=1,),margin=dict(l=130, r=20, t=100, b=120), width=1000, height=800, title="Mean and One Standard Deviation of Best Product Concentration", title_font_size=30)
fig.update_xaxes(title_text="Number of Experiments", title_font_size=30, tickfont=dict(size=30), range=[10,41])
fig.update_yaxes(title_text="Best Product Concentration", title_font_size=30, tickfont=dict(size=30),dtick=200)
fig.show()
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.write_image(f"{PDF_FOLDER}/best_objective_constrained.pdf")

2026-10-05 15:52:33,123 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 15:52:33,125 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 15:52:33,125 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpxlzkfk0m.
2026-10-05 15:52:33,126 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 15:52:33,127 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp4s34f6j6.
2026-10-05 15:52:33,127 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp4s34f6j6
2026-10-05 15:52:33,535 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpxlzkfk0m/index.html
2026-10-05 15:52:34,136 - kaleido.kaleido - INFO - Getting tab from queue 

In [13]:
fig = go.Figure()

for i, algo in enumerate(algorithms):
    color = color_dict[algo]

    upper = mean_violations_df[algo] + std_violations_df[algo]
    lower = mean_violations_df[algo] - std_violations_df[algo]
    if algo == "lhs":
        fig.add_trace(
            go.Scatter(
                x=mean_violations_df["Experiments"],
                y=mean_violations_df[algo],
                name=name_dict[algo],
                mode="markers",
                marker=dict(size=MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
                error_y=dict(
                    type="data",
                    array=std_violations_df[algo],
                    visible=True,
                    color=color,
                    thickness=2,
                    width=10,
                ),
                legendgroup=algo,
            )
        )
        
    else:
        fig.add_trace(
            go.Scatter(
                x=mean_violations_df["Experiments"],
                y=upper,
                line=dict(width=0),
                showlegend=False,
                hoverinfo="skip",
                legendgroup=algo,
            )
        )
        fig.add_trace(
            go.Scatter(
                x=mean_violations_df["Experiments"],
                y=lower,
                line=dict(width=0),
                fill="tonexty",
                fillcolor=_hex_to_rgba(color, 0.2),
                showlegend=False,
                hoverinfo="skip",
                legendgroup=algo,
            )
        )
        fig.add_trace(
            go.Scatter(
                x=mean_violations_df["Experiments"],
                y=mean_violations_df[algo],
                name=name_dict[algo],
                mode="lines+markers",
                line=dict(width=2, color=color),
                marker=dict(size=MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
                showlegend=False,
                legendgroup=algo,
            )
        )
        # legend-only trace so the legend line is thicker than the plotted line
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                name=name_dict[algo],
                mode="lines+markers",
                line=dict(width=6, color=color),
                marker=dict(size=LEGEND_MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
                legendgroup=algo,
            )
        )

fig.update_layout(legend=dict(bgcolor="rgba(255,255,255,0.5)",groupclick="togglegroup",font=dict(size=30), x=0.02, y=1,),margin=dict(l=130, r=20, t=100, b=120), width=1000, height=800, title="Mean and One Standard Deviation of Constraint Violations - Purity", title_font_size=30)
fig.update_xaxes(title_text="Number of Experiments", title_font_size=30, tickfont=dict(size=30), range=[10,41])
fig.update_yaxes(title_text="Constraint Violations - Purity", title_font_size=30, tickfont=dict(size=30))
fig.show()
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.write_image(f"{PDF_FOLDER}/constraint_violations.pdf")

2026-10-05 15:52:34,460 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 15:52:34,461 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 15:52:34,461 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpvqv77kug.
2026-10-05 15:52:34,462 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 15:52:34,463 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpa_k6ufal.
2026-10-05 15:52:34,464 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpa_k6ufal
2026-10-05 15:52:34,901 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpvqv77kug/index.html
2026-10-05 15:52:35,509 - kaleido.kaleido - INFO - Getting tab from queue 